# 01 · The data (Section 3)

This notebook shows what the analysis starts from. Nothing here is computed on a GPU; every file is in `data/`.

**Two corpora with human scores.**
* **IndicMT Eval**: English translated into Gujarati, Tamil, Malayalam, Marathi and Hindi; 1,400 segments per language; six MT systems; human MQM scores.
* **WMT24 and WMT25**: the general translation tasks; fifteen further settings with ESA or MQM scores.

A *setting* is one target language in one corpus. The paper uses twenty settings (Table 1).

In [ ]:
import os
import sys
import warnings
from pathlib import Path

# quieten library notices that do not affect any result
os.environ["TRANSFORMERS_VERBOSITY"] = "error"      # "no PyTorch found": no model is needed here
warnings.filterwarnings("ignore", module="tqdm")
warnings.filterwarnings("ignore", module="transformers")

import pandas as pd

ROOT = Path.cwd().resolve().parent          # the repository root
sys.path.insert(0, str(ROOT / "src"))
TABLES = ROOT / "results" / "tables"
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.precision", 3)

def table(name):
    """A table written by the analysis, from results/tables/."""
    return pd.read_csv(TABLES / f"{name}.csv")

## IndicMT Eval

Each segment appears twice: as released (`native`) and written in Latin script (`romanised`). The human score is the same in both rows, because annotators rated the native text once.

In [ ]:
seg = pd.read_parquet(ROOT / "data" / "indic" / "segments.parquet")
print(len(seg), "rows =", seg.segment_id.nunique(), "segments x 2 conditions")
print(seg[seg.condition == "native"].mqm_valid.sum(), "segments have a numeric human score (the working set)")
seg.groupby(["lang", "condition"], observed=True).size().unstack()

In [ ]:
seg[seg.condition == "native"].groupby("system", observed=True).size().rename("segments")

One segment in both scripts:

In [ ]:
ex = seg[seg.segment_id == seg.segment_id.iloc[0]][["condition", "source_en", "target", "mqm_score", "comet_22"]]
ex

## The twenty settings of Table 1

The re-segmentation experiment (notebook 02) runs on every setting below. The WMT systems that wrote the wrong language variety (for example Modern Standard Arabic instead of Egyptian Arabic) are dropped; they are listed after the settings.

In [ ]:
from tokinv import resegmentation
pd.DataFrame([s[:3] for s in resegmentation.SETTINGS if s[0] != "WMT25 CJK"],
             columns=["corpus", "human scores", "setting"])

In [ ]:
off = pd.read_csv(ROOT / "data" / "derived" / "offtarget" / "offtarget_systems.csv")
off[off.off_target]

## What "re-segmentation" means

A tokeniser cuts text into pieces from its vocabulary. The same string can be cut in many valid ways. Below, the romanised Gujarati word *jaheratanum* ('of the announcement') is cut the tokeniser's own way, then three other ways. Every cut decodes to exactly the same string. This is the intervention of the whole paper: only the cut changes, never the text.

In [ ]:
import random
from tokinv import sampler, tokenisers

tok = tokenisers.get("xlmr")
word = "jaheratanum"
print("tokeniser's own cut:", tok.tokenize(word))
for seed in range(3):
    s = sampler.sample_bucket(word, "xlmr", 1.5, 2.25, random.Random(seed))
    print("another valid cut:  ", tok.convert_ids_to_tokens(s["ids"]), "->", tok.decode(s["ids"]))
cuts = sampler.build_mdds(word, "xlmr")[0].enumerate_all()   # every valid cut of the word
print(f"{len(cuts):,} valid cuts in all, the tokeniser's own included")